# Model 02 — Naive Bayes (Surface Defect Risk)

This notebook is a **practical machine learning case study** focused on cosmetic defects in painted plastic parts.

A common real-world pain point is that visual defects rarely come from a single cause. A quality or process engineer typically has to connect signals across injection, storage/handling, and painting to understand risk drivers.  
Naive Bayes is used here as a **probabilistic baseline**: not to claim certainty, but to estimate defect risk under different operating conditions.

---

Data source: `surface_defect_inspection_data.csv`  
Output: metrics + statistical validation + digital Pareto + simulator + what‑if scenarios


## 1. Setup

We start by importing the libraries used throughout the notebook.  
The goal is reproducibility and clarity: a small, standard stack is enough for a solid baseline analysis.


In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, RocCurveDisplay
)

## 2. Load data

We load the dataset from a CSV file located in the repository folder.  
At this stage we only validate that the file loads correctly and inspect a few rows to confirm the schema.


In [ ]:
# Load the dataset from the repository folder
DATA_PATH = "surface_defect_inspection_data.csv"

df = pd.read_csv(DATA_PATH)
df.head()

## 3. Quick sanity checks

Before modeling, we run basic checks to understand:
- dataset size and column types,
- class balance (Pass/Fail),
- obvious missing values or formatting issues.

This prevents avoidable errors later in the pipeline.


In [ ]:
df.info()

In [ ]:
df["Surface_defect"].value_counts(dropna=False)

In [ ]:
df["Surface_defect"].value_counts(normalize=True).round(3)

## 4. Define X/y and column groups

We separate:
- **X**: process variables (numeric + categorical),
- **y**: the inspection outcome (Pass/Fail).

We also define column groups to apply the right preprocessing steps to each type of feature.


In [ ]:
# Target
y = df["Surface_defect"]

# Features
X = df.drop("Surface_defect", axis=1)

# Numeric features
numeric_features = [
    "Regrind_pct",
    "Resin_temp_C",
    "Cooling_time_s",
    "Paint_viscosity",
    "Film_thickness_um",
    "Booth_humidity_pct",
    "Prepaint_storage_time_h",
    "Handling_moves"
]

# Categorical features
categorical_features = [
    "Container_type",
    "Part_protection"
]

# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

## 5. Preprocessing + model pipeline (Naive Bayes)

Naive Bayes is a strong baseline when you want fast, probabilistic screening.  
In this section we build a pipeline that:
- one-hot encodes categorical features,
- keeps numeric features as-is,
- trains a Gaussian Naive Bayes classifier.

Using a pipeline ensures that preprocessing is applied consistently to train and test data.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline

# Column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(drop="first", sparse_output=False), categorical_features)
    ]
)

nb_model = Pipeline(steps=[
    ("prepro", preprocessor),
    ("clf", GaussianNB())
])

# Train
nb_model.fit(X_train, y_train)

## 6. Model evaluation

We evaluate the model using standard classification metrics.  
The focus is not only accuracy, but also how the model behaves on the minority class (defects), which is often the operational priority.


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = nb_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Accuracy (test): {acc:.3f}")
print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## 7. ROC curve and AUC

ROC-AUC summarizes how well the model separates Pass vs Fail across thresholds.  
This is useful when you might later tune the decision threshold based on scrap/rework strategy.


In [ ]:
from sklearn.metrics import roc_auc_score, RocCurveDisplay
import matplotlib.pyplot as plt

y_proba = nb_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_proba)

print(f"ROC-AUC (test): {auc:.3f}")

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("ROC Curve — Naive Bayes (Surface Defect)")
plt.show()

## 8. Statistical validation (95% CI + cross-validation)

Single train/test splits can be noisy.  
Here we estimate uncertainty with:
- confidence intervals for key metrics,
- cross-validation scores to check stability.

This helps avoid over-interpreting one lucky split.


In [ ]:
import numpy as np

n_test = len(y_test)
se = np.sqrt(acc * (1 - acc) / n_test)
z = 1.96  # 95%

ci_low = acc - z * se
ci_high = acc + z * se

print(f"Accuracy (test): {acc:.3f}")
print(f"95% CI for Accuracy: [{ci_low:.3f}, {ci_high:.3f}]")

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(nb_model, X_train, y_train, cv=5, scoring="accuracy")

print("CV Accuracy scores:", np.round(cv_scores, 3))
print(f"Mean ± Std: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

## 9. Effect size (Cohen’s d) for numeric features + digital Pareto

Beyond model metrics, we want to understand **which variables shift most** between Pass and Fail.  
We compute effect size for numeric features and summarize the strongest drivers in a Pareto-style view.


In [ ]:
import numpy as np
import pandas as pd

def cohens_d(group0, group1):
    g0 = np.asarray(group0)
    g1 = np.asarray(group1)
    n0, n1 = len(g0), len(g1)

    s0 = g0.std(ddof=1)
    s1 = g1.std(ddof=1)

    sp = np.sqrt(((n0 - 1) * s0**2 + (n1 - 1) * s1**2) / (n0 + n1 - 2))
    if sp == 0 or np.isnan(sp):
        return 0.0
    return (g1.mean() - g0.mean()) / sp

# Use the SAME lists you already defined
# numeric_features, categorical_features

d_scores = {}
for col in numeric_features:
    pass_vals = df[df["Surface_defect"] == 0][col]
    fail_vals = df[df["Surface_defect"] == 1][col]
    d_scores[col] = abs(cohens_d(pass_vals, fail_vals))

d_scores = pd.Series(d_scores).sort_values(ascending=False)
d_scores

In [ ]:
from sklearn.feature_selection import mutual_info_classif

X_all = df.drop("Surface_defect", axis=1)
y_all = df["Surface_defect"]

X_dum = pd.get_dummies(X_all, drop_first=True)

mi = mutual_info_classif(X_dum, y_all, random_state=42)
mi_scores = pd.Series(mi, index=X_dum.columns).sort_values(ascending=False)

mi_scores.head(20)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Normalize blocks separately (0–1)
d_norm = d_scores / (d_scores.max() if d_scores.max() > 0 else 1)
mi_norm = mi_scores / (mi_scores.max() if mi_scores.max() > 0 else 1)

combined = pd.concat([d_norm, mi_norm]).sort_values(ascending=False)

# Scale 0–100 for visualization
pareto = (combined / combined.max() * 100).round(2)

# Proper cumulative line
vals = pareto.values
labels = pareto.index

cum = np.cumsum(vals)
cum = cum / cum[-1] * 100

plt.figure(figsize=(14, 6))
plt.bar(labels, vals)
plt.xticks(rotation=75, ha="right")
plt.plot(labels, cum, marker="o")
plt.axhline(80, linestyle="--")
plt.title("Digital Pareto — Drivers of Surface Defects")
plt.ylabel("Importance (%) / Cumulative (%)")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

## 10. Simulator (what-if tool)

This section provides a lightweight scenario tool.  
You can input hypothetical conditions (injection / painting / handling) and get an estimated defect probability.

The intent is to support operational conversations, not to “predict the future”.


In [ ]:
import pandas as pd

def simulate_surface_defect_risk(
    model_pipeline,
    Regrind_pct: float,
    Resin_temp_C: float,
    Cooling_time_s: float,
    Paint_viscosity: float,
    Film_thickness_um: float,
    Booth_humidity_pct: float,
    Prepaint_storage_time_h: float,
    Handling_moves: int,
    Container_type: str,
    Part_protection: str,
    threshold: float = 0.50
):
    """
    What-if simulator:
    Returns defect probability and Pass/Fail decision based on a configurable threshold.
    """

    row = pd.DataFrame([{
        "Regrind_pct": Regrind_pct,
        "Resin_temp_C": Resin_temp_C,
        "Cooling_time_s": Cooling_time_s,
        "Paint_viscosity": Paint_viscosity,
        "Film_thickness_um": Film_thickness_um,
        "Booth_humidity_pct": Booth_humidity_pct,
        "Prepaint_storage_time_h": Prepaint_storage_time_h,
        "Handling_moves": Handling_moves,
        "Container_type": Container_type,
        "Part_protection": Part_protection
    }])

    prob_fail = model_pipeline.predict_proba(row)[0, 1]
    decision = "FAIL (High Risk)" if prob_fail >= threshold else "PASS (Low/Moderate Risk)"
    return prob_fail, decision

In [ ]:
# Good conditions
prob, decision = simulate_surface_defect_risk(
    nb_model,
    Regrind_pct=5,
    Resin_temp_C=235,
    Cooling_time_s=20,
    Paint_viscosity=25,
    Film_thickness_um=30,
    Booth_humidity_pct=50,
    Prepaint_storage_time_h=1,
    Handling_moves=2,
    Container_type="plastic_box",
    Part_protection="with_protection",
    threshold=0.50
)

print(f"Defect probability: {prob:.3f} | Decision: {decision}")

In [ ]:
# Bad handling + bad enviroment :(
prob, decision = simulate_surface_defect_risk(
    nb_model,
    Regrind_pct=28,
    Resin_temp_C=248,
    Cooling_time_s=13,
    Paint_viscosity=21,
    Film_thickness_um=38,
    Booth_humidity_pct=68,
    Prepaint_storage_time_h=18,
    Handling_moves=6,
    Container_type="metal_rack",
    Part_protection="no_protection",
    threshold=0.50
)

print(f"Defect probability: {prob:.3f} | Decision: {decision}")

## 11. Quick top drivers

We surface the most influential variables identified in this notebook and translate them into operational language (what to watch, what to control).


In [ ]:
top_10 = pareto.head(10).reset_index()
top_10.columns = ["Feature", "Importance_Score"]
top_10

## Final Reflection
This project is not about predicting defects with certainty.
It is about making cosmetic risk visible.

Surface defects in painted plastic parts are rarely the result of a single mistake.
They emerge from interactions across the process: injection conditions, storage time, handling intensity, protection practices, and paint shop environment.

Naive Bayes works particularly well in this type of problem because it treats defects as a probabilistic outcome, not as a deterministic failure.
It allows us to combine information coming from different stages of the process and translate it into an interpretable risk.

There is no single variable that “causes” a surface defect.
Defects appear when small deviations accumulate and reveal themselves at the surface.

What this model offers is not a verdict, but a conversation:

What happens if parts stay longer in storage before painting?
How much cosmetic risk do we add with each additional handling step?
Is protection more critical than tightening paint parameters?
Which levers actually reduce scrap, and which ones just feel important?

The simulator reinforces this idea.
Same model. Different scenarios. Different risks.

And that is the real value of Naive Bayes in manufacturing quality:
not magic, not certainty,
just probabilities that help teams understand risk, challenge assumptions, and improve the process step by step.

—
Not magic. Just probabilities.  
**Where f(x) meets Kaizen**  
LozanoLsa